This notebook processes multiple CSV files located in a specified Google Drive folder. It performs the following steps for each file:
1.  Connects to Google Drive.
2.  Defines input and output folder paths.
3.  Lists all CSV files in the input folder.
4.  Reads each CSV file into a pandas DataFrame.
5.  Renames specific columns ('영업일자' to 'date', '영업장명_메뉴명' to 'store_menu', '매출수량' to 'sales').
6.  Splits the 'store_menu' column into 'store' and 'menu' columns.
7.  Converts the 'date' column to datetime objects and creates a 'date_ordinal' column representing the ordinal date.
8.  Reorders the columns to place 'date_ordinal' before 'date' and 'sales' at the end.
9.  Saves the processed DataFrame to the specified output folder with the original filename.

In [64]:
# 1. gdrive 연결
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [65]:
import os
import pandas as pd
from datetime import datetime

# Google Drive의 기본 경로를 설정합니다.
# 필요에 따라 'My Drive' 부분을 사용자 Drive 경로에 맞게 수정하세요.
base_drive_path = '/content/gdrive/My Drive/data_filtering'

# 원본 파일이 있는 폴더와 파일을 저장할 폴더 경로를 설정합니다.
original_folder_path = os.path.join(base_drive_path, 'original')
output_folder_path = os.path.join(base_drive_path, 'filtered') # 사용자 요청에 따라 filtered 폴더에 저장

# 출력 폴더가 없으면 생성합니다.
if not os.path.exists(output_folder_path):
    os.makedirs(output_folder_path)

print(f"원본 폴더: {original_folder_path}")
print(f"저장 폴더: {output_folder_path}")

원본 폴더: /content/gdrive/My Drive/data_filtering/original
저장 폴더: /content/gdrive/My Drive/data_filtering/filtered


In [66]:
# 2. original 폴더 안의 csv 파일 목록 가져오기
csv_files = [f for f in os.listdir(original_folder_path) if f.endswith('.csv')]

if not csv_files:
    print(f"'{original_folder_path}' 폴더에 CSV 파일이 없습니다.")
else:
    print(f"처리할 CSV 파일 목록: {csv_files}")

처리할 CSV 파일 목록: ['TEST_06.csv', 'TEST_03.csv', 'TEST_07.csv', 'TEST_04.csv', 'TEST_05.csv', 'TEST_01.csv', 'TEST_08.csv', 'TEST_00.csv', 'TEST_02.csv', 'TEST_09.csv', 'train.csv']


### Function: Process a single CSV file

This function encapsulates the processing steps for a single CSV file.

In [67]:
def process_csv_file(original_file_path, output_file_path):
    """
    Reads a CSV file, performs data transformations, and saves the processed file.

    Args:
        original_file_path (str): The full path to the original CSV file.
        output_file_path (str): The full path to save the processed CSV file.
    """
    file_name = os.path.basename(original_file_path)
    print(f"\nProcessing {file_name}...")

    try:
        # CSV 파일 읽기 (utf-8 인코딩 지정)
        df = pd.read_csv(original_file_path, encoding='utf-8')
        print(f"Original shape: {df.shape}")

        # 디버깅용 첫 행 출력
        if not df.empty:
            print("First row (for debugging):")
            display(df.head(1))
        else:
            print("DataFrame is empty.")

        # 컬럼명 변경
        new_column_names = {
            '영업일자': 'date',
            '영업장명_메뉴명': 'store_menu',
            '매출수량': 'sales'
        }
        df = df.rename(columns=new_column_names)
        print("Columns renamed.")

        # store_menu 컬럼 분리
        if 'store_menu' in df.columns:
            df[['store', 'menu']] = df['store_menu'].str.split('_', n=1, expand=True)
            print("'store_menu' column split into 'store' and 'menu'.")
        else:
            print("'store_menu' column not found.")

        # date 열을 datetime 객체로 변환하고 dateordinal 열 추가
        if 'date' in df.columns:
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df['date_ordinal'] = df['date'].map(lambda x: x.toordinal() if pd.notnull(x) else None)
            print("'date' column converted to datetime and 'date_ordinal' column added.")
        else:
            print("'date' column not found.")

        # sales 컬럼 음수 값 0으로 변경
        if 'sales' in df.columns:
            negative_sales_mask = df['sales'] < 0
            if negative_sales_mask.any():
                num_negative_sales = negative_sales_mask.sum()
                df.loc[negative_sales_mask, 'sales'] = 0
                print(f"Changed {num_negative_sales} negative sales values to 0.")
            else:
                print("No negative sales values found.")
        else:
            print("'sales' column not found.")


        # 원본 DataFrame을 filtered 폴더에 저장 (파일명 유지)
        # 이 부분은 monkey-patch된 함수에서 라벨 병합 후 최종 저장됩니다.
        print(f"Saving intermediate {file_name} to {output_folder_path}...")
        df.to_csv(output_file_path, index=False)
        print(f"Saved intermediate {file_name}")


    except FileNotFoundError:
        print(f"Error: File not found at {original_file_path}")
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

In [68]:
import numpy as np
import pandas as pd
from datetime import timedelta
try:
    import holidays
except Exception as e:
    holidays = None
    print("[WARN] 'holidays' is missing. Please: pip install holidays")

def _detect_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    low = [c.lower() for c in df.columns]
    for cand in candidates:
        if cand.lower() in low:
            return df.columns[low.index(cand.lower())]
    raise KeyError(f"Required column is missing. candidates={candidates}, df.columns={list(df.columns)[:20]}")

def _ensure_datetime(df, date_col):
    if not np.issubdtype(df[date_col].dtype, np.datetime64):
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    if df[date_col].isna().any():
        raw = df[date_col].astype(str).str.replace(r"[^0-9]", "", regex=True)
        try:
            df[date_col] = pd.to_datetime(raw, format="%Y%m%d", errors="coerce")
        except Exception:
            pass
    return df

def _season_from_month(m):
    # 0=봄(3-5), 1=여름(6-8), 2=가을(9-11), 3=겨울(12-2)
    if m in (3,4,5): return 0
    if m in (6,7,8): return 1
    if m in (9,10,11): return 2
    return 3

def _add_calendar_features(df, date_col):
    if holidays is None:
        raise ImportError("`holidays` package required. Run: pip install holidays")
    df = df.sort_values(date_col).reset_index(drop=True)

    years = list(range(int(df[date_col].dt.year.min()), int(df[date_col].dt.year.max())+1))
    kr_holidays = holidays.KR(years=years, observed=True)

    # base calendar
    df["dow"] = df[date_col].dt.weekday
    df["month"] = df[date_col].dt.month
    df["quarter"] = df[date_col].dt.quarter
    df["is_month_start"] = (df[date_col].dt.is_month_start).astype("int8")
    df["is_month_end"] = (df[date_col].dt.is_month_end).astype("int8")
    df["season"] = df["month"].apply(_season_from_month).astype("int8")

    # cyclic encodings
    doy = df[date_col].dt.dayofyear
    df["sin_doy"] = np.sin(2*np.pi * doy/365.25).astype("float32")
    df["cos_doy"] = np.cos(2*np.pi * doy/365.25).astype("float32")
    df["sin_dow"] = np.sin(2*np.pi * df["dow"]/7.0).astype("float32")
    df["cos_dow"] = np.cos(2*np.pi * df["dow"]/7.0).astype("float32")

    # weekends/holidays (keep only the final flags we want)
    df["is_weekend"] = df["dow"].isin([5,6]).astype("int8")
    # temp holiday/off computation for internal logic
    _is_holiday = df[date_col].apply(lambda d: 1 if d in kr_holidays else 0).astype("int8")
    _is_off = ((_is_holiday==1) | (df["is_weekend"]==1)).astype("int8")

    # keep is_holiday, drop holiday_name and others
    df["is_holiday"] = _is_holiday

    # Compute day_type WITHOUT leaving is_off/is_off_prev/is_off_next in the dataframe
    is_weekday = (_is_off.values==0)
    is_off_prev = np.roll(_is_off.values, 1); is_off_prev[0] = 0
    is_off_next = np.roll(_is_off.values, -1); is_off_next[-1] = 0

    day_type = np.full(len(df), 1, dtype="int8")
    sand = (is_weekday & (is_off_prev==1) & (is_off_next==1))
    day_type[sand] = 5
    cond3 = (is_weekday & (is_off_next==1))
    day_type[cond3 & ~sand] = 3
    cond4 = ((_is_off.values==1) & (is_off_next==0))
    day_type[cond4 & ~sand] = 4
    day_type[(_is_off.values==1) & ~(sand | cond4)] = 2
    df["day_type"] = day_type

    # Build off-run id and pos using _is_off internally
    off = _is_off.values
    run_id = np.zeros(len(off), dtype=int)
    rid = 0
    for i, v in enumerate(off):
        if i==0:
            rid = 1 if v==1 else 0
            run_id[i] = rid
        else:
            if v==1:
                if off[i-1]==1:
                    run_id[i] = rid
                else:
                    rid += 1
                    run_id[i] = rid
            else:
                run_id[i] = 0
    df["_off_run_id"] = run_id

    off_pos = np.zeros(len(df), dtype="int8")
    for idx in np.unique(df["_off_run_id"]):
        if idx==0: continue
        pos = np.flatnonzero(df["_off_run_id"].values==idx)
        if len(pos)==1:
            off_pos[pos[0]] = 3
        else:
            off_pos[pos[0]] = 1
            off_pos[pos[-1]] = 3
            if len(pos)>2:
                off_pos[pos[1:-1]] = 2
    df["off_run_pos"] = off_pos

    # dtype adjustments
    int8_cols = ["dow","month","quarter","season","is_weekend","is_holiday",
                 "day_type","off_run_pos","is_month_start","is_month_end"]
    for c in int8_cols:
        if c in df.columns:
            df[c] = df[c].astype("int8")
    return df

def build_peak_lookup(train_df, key_cols=("store_menu","month"), sales_col=None, q=0.80):
    if sales_col is None:
        sales_col = "sales" if "sales" in train_df.columns else _detect_col(train_df, ["매출수량","sales"])
    g = train_df.groupby(list(key_cols))[sales_col].median().reset_index(name="median_sales")
    thresh = g.groupby(key_cols[0])["median_sales"].transform(lambda s: s.quantile(q))
    g["is_peak_data"] = (g["median_sales"] >= thresh).astype("int8")
    return g[list(key_cols) + ["is_peak_data"]].copy()

def apply_peak_lookup(df, peak_lut, key_cols=("store_menu","month")):
    return df.merge(peak_lut, on=list(key_cols), how="left").assign(
        is_peak_data=lambda x: x["is_peak_data"].fillna(0).astype("int8")
    )

def add_domain_features(df, train_peak_lut=None):
    # detect date/store_menu columns
    date_col = _detect_col(df, ["date","영업일자"])
    df = _ensure_datetime(df.copy(), date_col)

    if "store_menu" not in df.columns:
        if "store" in df.columns and "menu" in df.columns:
            df["store_menu"] = df["store"].astype(str) + "_" + df["menu"].astype(str)
        else:
            sm = _detect_col(df, ["영업장명_메뉴명","store_menu"])
            df = df.rename(columns={sm: "store_menu"})

    # calendar/holiday/run features
    df = _add_calendar_features(df, date_col)

    # data-based peak
    if train_peak_lut is not None:
        df = apply_peak_lookup(df, train_peak_lut, key_cols=("store_menu","month"))
    else:
        df["is_peak_data"] = np.int8(0)

    return df

# --- safe base wrapping (no recursion) ---
try:
    _base_process_csv_file
except NameError:
    _base_process_csv_file = process_csv_file

def process_csv_file(original_file_path, output_file_path, peak_lut_for_inference=None):
    """
    Run the existing preprocessing, then enrich with selected calendar features + data-peak.
    Excluded columns (never created): holiday_name, is_off, is_off_prev, is_off_next, off_run_len,
                                     days_since_prev_off, days_to_next_off, is_major_holiday,
                                     is_before_major_holiday, is_after_major_holiday
    """
    _base_process_csv_file(original_file_path, output_file_path)

    try:
        df = pd.read_csv(output_file_path)
    except UnicodeDecodeError:
        df = pd.read_csv(output_file_path, encoding="cp949")

    df = add_domain_features(df, train_peak_lut=peak_lut_for_inference)
    df.to_csv(output_file_path, index=False)
    print(f"[OK] v3 calendar features added. Columns: {df.shape[1]} → {output_file_path}")

In [69]:
# === LABEL CONFIG (add-on) ===
# 라벨 CSV 경로를 환경에 맞게 설정하세요.
LABELS_PATH = '/content/gdrive/My Drive/data_filtering/menu_labels_for_patchtst.csv'

try:
    labels_df = pd.read_csv(LABELS_PATH)
    LABEL_COLS = ['menu_cluster','menu_cluster_label','pattern_group','pattern_group_label']
    labels_df['store'] = labels_df['store'].astype(str).str.strip()
    labels_df['menu']  = labels_df['menu'].astype(str).str.strip()
    _miss = [c for c in ['store','menu'] + LABEL_COLS if c not in labels_df.columns]
    if _miss:
        raise ValueError(f"[LABEL ERROR] 필요한 컬럼 누락: {_miss}")
    if labels_df.duplicated(['store','menu']).any():
        raise ValueError("[LABEL ERROR] (store,menu) 중복 라벨이 존재합니다.")
    print(f"[LABEL] 라벨 로드 완료: {labels_df.shape}, path={LABELS_PATH}")
    MERGE_LABELS_ENABLED = True
    STRICT_LABEL_CHECK = False
except Exception as e:
    print(f"[LABEL WARNING] 라벨 파일을 로드하지 못했습니다: {e}")
    labels_df = None
    LABEL_COLS = ['menu_cluster','menu_cluster_label','pattern_group','pattern_group_label']
    MERGE_LABELS_ENABLED = False
    STRICT_LABEL_CHECK = False


[LABEL] 라벨 로드 완료: (193, 6), path=/content/gdrive/My Drive/data_filtering/menu_labels_for_patchtst.csv


In [70]:
# === Monkey-patch: process_csv_file with label merge ===
# 기존 process_csv_file을 보존하고, 동일 이름으로 래핑하여
# 원래 기능 수행 후 라벨을 병합합니다.

# 백업
_original_process_csv_file = process_csv_file

def process_csv_file(original_file_path, output_file_path, *args, **kwargs):
    """Wrapper: 원래 전처리 수행 후, 라벨 4종을 병합해서 저장"""
    # 1) 기존 처리 수행 (원본 기능 유지)
    # 원본 함수에서 중간 저장이 일어나지만, 아래 라벨 병합 후 최종 저장으로 덮어쓰여집니다.
    _original_process_csv_file(original_file_path, output_file_path, *args, **kwargs)

    # 2) 라벨 병합 수행
    if not MERGE_LABELS_ENABLED or labels_df is None:
        print("[LABEL] 라벨 병합 비활성화 또는 라벨 미로딩 → 스킵.")
        return

    try:
        # 원본 함수에서 저장된 중간 파일을 다시 읽어옵니다.
        df = pd.read_csv(output_file_path, encoding='utf-8')

        # store/menu 확보 (이미 있으면 그대로 사용)
        has_store = 'store' in df.columns
        has_menu  = 'menu'  in df.columns

        if not (has_store and has_menu):
            if 'store_menu' in df.columns:
                tmp = df['store_menu'].astype(str).str.split('_', n=1, expand=True)
                df['store'] = tmp[0].astype(str).str.strip()
                df['menu']  = tmp[1].fillna('').astype(str).str.strip()
                has_store, has_menu = True, True
                print("[LABEL] 'store_menu' → 'store','menu' 분리 완료.")
            else:
                print("[LABEL] 'store'+'menu' 또는 'store_menu'가 없어 라벨 병합 스킵.")
                df.to_csv(output_file_path, index=False, encoding='utf-8')
                return

        # 키 전처리
        df['store'] = df['store'].astype(str).str.strip()
        df['menu']  = df['menu'].astype(str).str.strip()

        # 기존 라벨 컬럼이 있으면 중복 방지를 위해 제거
        dup_cols = [c for c in LABEL_COLS if c in df.columns]
        if dup_cols:
            df = df.drop(columns=dup_cols)
            print(f"[LABEL] 기존 라벨 컬럼 제거: {dup_cols}")

        before_cols = set(df.columns)

        merged = df.merge(
            labels_df[['store','menu'] + LABEL_COLS],
            on=['store','menu'],
            how='left',
            validate='m:1',
            suffixes=('', '_dup')
        )

        # 미매칭 확인
        unmatched_mask = merged['menu_cluster'].isna() | merged['menu_cluster_label'].isna() | \
                         merged['pattern_group'].isna() | merged['pattern_group_label'].isna()
        n_unmatched = int(unmatched_mask.sum())
        if n_unmatched > 0:
            sample = merged.loc[unmatched_mask, ['store','menu']].drop_duplicates().head(10)
            msg = f"[LABEL WARNING] 라벨 미매칭 {n_unmatched}행. 예시:\n{sample}"
            if STRICT_LABEL_CHECK:
                raise ValueError(msg)
            else:
                print(msg)
        else:
            print("[LABEL] 라벨 병합 완료 (모든 행 매칭).")

        new_cols = [c for c in LABEL_COLS if c in merged.columns and c not in before_cols]
        print(f"[LABEL] 추가된 라벨 컬럼: {new_cols}")

        # --- 최종 컬럼 순서 조정 (저장 직전에 수행) ---
        # 원하는 컬럼 순서 정의
        desired_order = [
            'date_ordinal', 'date', 'store', 'menu', 'store_menu',
            'menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label',
            'sales'
        ]

        # 현재 DataFrame의 컬럼 목록
        current_cols = merged.columns.tolist()

        # 원하는 순서의 컬럼 중 현재 DataFrame에 존재하는 컬럼만 추출하여 순서대로 나열합니다.
        ordered_cols = [col for col in desired_order if col in current_cols]

        # 원하는 순서에 포함되지 않는 나머지 컬럼들을 현재 DataFrame에서의 순서대로 나열합니다.
        remaining_cols = [col for col in current_cols if col not in desired_order]

        # 최종 컬럼 순서를 조합합니다.
        final_col_order = ordered_cols + remaining_cols

        # DataFrame의 컬럼 순서를 재정렬합니다.
        merged = merged[final_col_order]
        print("[LABEL] 최종 컬럼 순서 재조정 완료.")
        # --- 최종 컬럼 순서 조정 끝 ---


        merged.to_csv(output_file_path, index=False, encoding='utf-8')
        print("[LABEL] 병합 결과 저장 완료.")

    except Exception as e:
        print(f"[LABEL ERROR] 라벨 병합 중 오류: {e}")

### Main Processing Loop

This block iterates through the list of CSV files and calls the `process_csv_file` function for each one.

In [71]:
# 파일 처리 및 저장 루프
processed_files_count = 0
for file_name in csv_files:
    original_file_path = os.path.join(original_folder_path, file_name)
    output_file_path = os.path.join(output_folder_path, file_name) # 원본 파일명 그대로 저장
    process_csv_file(original_file_path, output_file_path)
    processed_files_count += 1

print(f"\n모든 파일 처리 시도 완료. 총 {processed_files_count}개의 파일 처리가 완료되었습니다.")


Processing TEST_06.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2025-01-12,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_06.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_06.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_06.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Processing TEST_03.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2024-09-29,느티나무 셀프BBQ_1인 수저세트,5


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_03.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_03.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_03.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Processing TEST_07.csv...
Original shape: (5404, 3)
First row (for debugging):


...

Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_02.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_02.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_02.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Processing TEST_09.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2025-04-27,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_09.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_09.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_09.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Processing train.csv...
Original shape: (102676, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2023-01-01,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate train.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate train.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/train.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

모든 파일 처리 시도 완료. 총 11개의 파일 처리가 완료되었습니다.


# Task
Extend the test datasets by 7 days and generate features for the extended dates, excluding the 'sales' column.

## Identify test files

### Subtask:
Modify the main processing loop to identify files that are part of the 'test' set (e.g., starting with 'TEST_').


**Reasoning**:
Modify the existing loop to check if the file name starts with 'TEST_'.



In [72]:
# 파일 처리 및 저장 루프
processed_files_count = 0
for file_name in csv_files:
    original_file_path = os.path.join(original_folder_path, file_name)
    output_file_path = os.path.join(output_folder_path, file_name) # 원본 파일명 그대로 저장

    # Check if the file is a test file
    is_test_file = file_name.startswith('TEST_')

    # Now you can use the 'is_test_file' variable within the loop
    # For this subtask, just print the status for demonstration
    print(f"\nChecking file: {file_name}. Is test file? {is_test_file}")

    # The actual processing call is commented out for now as per the subtask
    # process_csv_file(original_file_path, output_file_path)

    processed_files_count += 1

print(f"\n모든 파일 처리 시도 완료. 총 {processed_files_count}개의 파일 처리가 완료되었습니다.")


Checking file: TEST_06.csv. Is test file? True

Checking file: TEST_03.csv. Is test file? True

Checking file: TEST_07.csv. Is test file? True

Checking file: TEST_04.csv. Is test file? True

Checking file: TEST_05.csv. Is test file? True

Checking file: TEST_01.csv. Is test file? True

Checking file: TEST_08.csv. Is test file? True

Checking file: TEST_00.csv. Is test file? True

Checking file: TEST_02.csv. Is test file? True

Checking file: TEST_09.csv. Is test file? True

Checking file: train.csv. Is test file? False

모든 파일 처리 시도 완료. 총 11개의 파일 처리가 완료되었습니다.


## Load and extend test data

### Subtask:
For identified test files, load the processed DataFrame and extend the 'date' column by 7 days, ensuring the 'date_ordinal' is also extended accordingly.


**Reasoning**:
Load the processed test file, identify unique store/menu combinations, find the last date for each, generate the next 7 dates, create new rows, and concatenate them to the DataFrame.



In [73]:
# File processing and saving loop
processed_files_count = 0
for file_name in csv_files:
    original_file_path = os.path.join(original_folder_path, file_name)
    output_file_path = os.path.join(output_folder_path, file_name) # Save with original filename

    # Check if the file is a test file
    is_test_file = file_name.startswith('TEST_')

    # Process the file (including the original steps + calendar features + label merge)
    process_csv_file(original_file_path, output_file_path)

    if is_test_file:
        print(f"\nExtending test file: {file_name}")
        try:
            # Load the already processed DataFrame for the test file
            df = pd.read_csv(output_file_path, encoding='utf-8')
            print(f"Loaded processed test file: {file_name}")
            print(f"Original shape before extension: {df.shape}")

            # Ensure 'date' is datetime and 'date_ordinal' is correct
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df['date_ordinal'] = df['date'].map(lambda x: x.toordinal() if pd.notnull(x) else None)

            # Identify unique store/menu combinations
            unique_combinations = df[['store', 'menu']].drop_duplicates()

            extended_rows = []

            # For each unique combination, find the last date and generate next 7 days
            for index, row in unique_combinations.iterrows():
                store = row['store']
                menu = row['menu']

                # Filter for the current store/menu combination
                subset_df = df[(df['store'] == store) & (df['menu'] == menu)].copy()

                # Find the last date for this combination
                if not subset_df.empty:
                    last_date = subset_df['date'].max()

                    # Generate the next 7 dates
                    next_dates = [last_date + timedelta(days=i) for i in range(1, 8)]

                    # Create new rows for these dates
                    for next_date in next_dates:
                        new_row = {
                            'date': next_date,
                            'date_ordinal': next_date.toordinal(),
                            'store': store,
                            'menu': menu,
                            'store_menu': f"{store}_{menu}",
                            'sales': np.nan # Sales is unknown for future dates
                            # Other columns will be added/filled later by add_domain_features and label merge
                        }
                        extended_rows.append(new_row)

            if extended_rows:
                # Create a DataFrame from the new rows
                extended_df = pd.DataFrame(extended_rows)

                # Concatenate the new rows to the original DataFrame
                df = pd.concat([df, extended_df], ignore_index=True)

                # Ensure date and date_ordinal are correct after concatenation
                df['date'] = pd.to_datetime(df['date'], errors='coerce')
                df['date_ordinal'] = df['date'].map(lambda x: x.toordinal() if pd.notnull(x) else None)

                print(f"Extended {file_name} by 7 days for each store/menu combination.")
                print(f"Shape after extension: {df.shape}")

                # Save the extended DataFrame (intermediate save before adding other features)
                df.to_csv(output_file_path, index=False, encoding='utf-8')
                print(f"Saved extended intermediate {file_name}")

            else:
                print(f"No data to extend for {file_name}.")


        except Exception as e:
            print(f"Error extending test file {file_name}: {e}")


    processed_files_count += 1

print(f"\n모든 파일 처리 시도 완료. 총 {processed_files_count}개의 파일 처리가 완료되었습니다.")


Processing TEST_06.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2025-01-12,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_06.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_06.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_06.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending test file: TEST_06.csv
Loaded processed test file: TEST_06.csv
Original shape before extension: (5404, 26)
Extended TEST_06.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Saved extended intermediate TEST_06.csv

Processing TEST_03.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2024-09-29,느티나무 셀프BBQ_1인 수저세트,5


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_03.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_03.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_03.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending test file: TEST_03.csv
Loaded processed test file: TEST_03.csv
Original shape before extension: (5404, 26)
Extended TEST_03.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Saved extended intermediate TEST_03.csv

Processing TEST_07.csv...
Original shape: (5404, 3)
First row (for debugging):


...

Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_02.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_02.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_02.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending test file: TEST_02.csv
Loaded processed test file: TEST_02.csv
Original shape before extension: (5404, 26)
Extended TEST_02.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Saved extended intermediate TEST_02.csv

Processing TEST_09.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2025-04-27,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_09.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_09.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_09.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending test file: TEST_09.csv
Loaded processed test file: TEST_09.csv
Original shape before extension: (5404, 26)
Extended TEST_09.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Saved extended intermediate TEST_09.csv

Processing train.csv...
Original shape: (102676, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2023-01-01,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate train.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate train.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/train.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

모든 파일 처리 시도 완료. 총 11개의 파일 처리가 완료되었습니다.


## Generate features for extended dates

### Subtask:
Apply the existing `add_domain_features` function to the extended DataFrame to generate calendar and other relevant features for the new dates.


**Reasoning**:
Apply the existing `add_domain_features` function to the extended DataFrame to generate calendar and other relevant features for the new dates, then save the updated DataFrame.



In [74]:
# File processing and saving loop
processed_files_count = 0
for file_name in csv_files:
    original_file_path = os.path.join(original_folder_path, file_name)
    output_file_path = os.path.join(output_folder_path, file_name) # Save with original filename

    # Check if the file is a test file
    is_test_file = file_name.startswith('TEST_')

    # Process the file (including the original steps + calendar features + label merge)
    # The monkey-patched process_csv_file already handles the initial processing and label merge
    process_csv_file(original_file_path, output_file_path)

    if is_test_file:
        print(f"\nExtending test file: {file_name}")
        try:
            # Load the already processed DataFrame for the test file
            # Reloading to ensure the previous processing steps are included
            df = pd.read_csv(output_file_path, encoding='utf-8')
            print(f"Loaded processed test file: {file_name}")
            print(f"Original shape before extension: {df.shape}")

            # Ensure 'date' is datetime and 'date_ordinal' is correct
            # This might be redundant if process_csv_file works correctly, but serves as a safeguard
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df['date_ordinal'] = df['date'].map(lambda x: x.toordinal() if pd.notnull(x) else None)

            # Identify unique store/menu combinations
            unique_combinations = df[['store', 'menu']].drop_duplicates()

            extended_rows = []

            # For each unique combination, find the last date and generate next 7 days
            for index, row in unique_combinations.iterrows():
                store = row['store']
                menu = row['menu']

                # Filter for the current store/menu combination
                subset_df = df[(df['store'] == store) & (df['menu'] == menu)].copy()

                # Find the last date for this combination
                if not subset_df.empty:
                    last_date = subset_df['date'].max()

                    # Generate the next 7 dates
                    next_dates = [last_date + timedelta(days=i) for i in range(1, 8)]

                    # Create new rows for these dates
                    for next_date in next_dates:
                        new_row = {
                            'date': next_date,
                            'date_ordinal': next_date.toordinal(),
                            'store': store,
                            'menu': menu,
                            'store_menu': f"{store}_{menu}",
                            'sales': np.nan # Sales is unknown for future dates
                            # Other columns will be added/filled later by add_domain_features and label merge
                        }
                        extended_rows.append(new_row)

            if extended_rows:
                # Create a DataFrame from the new rows
                extended_df = pd.DataFrame(extended_rows)

                # Concatenate the new rows to the original DataFrame
                df = pd.concat([df, extended_df], ignore_index=True)

                # Ensure date and date_ordinal are correct after concatenation
                df['date'] = pd.to_datetime(df['date'], errors='coerce')
                df['date_ordinal'] = df['date'].map(lambda x: x.toordinal() if pd.notnull(x) else None)

                print(f"Extended {file_name} by 7 days for each store/menu combination.")
                print(f"Shape after extension: {df.shape}")

                # --- Subtask 3: Apply add_domain_features ---
                # Apply the add_domain_features function to the extended dataframe
                # Assuming labels_df (global variable) contains the necessary lookup data
                # If train_peak_lut is needed, it should be generated from the training data (e.g., 'train.csv')
                # For this subtask, we apply add_domain_features without train_peak_lut as specified.
                print("Applying add_domain_features to the extended DataFrame...")
                df = add_domain_features(df, train_peak_lut=None) # Apply features to both original and extended dates
                print(f"Applied add_domain_features. New shape: {df.shape}")
                # --- End Subtask 3 ---

                # Reapply labels to the combined data, as add_domain_features might not carry them over
                if MERGE_LABELS_ENABLED and labels_df is not None:
                    print("[LABEL] Re-merging labels after adding domain features...")
                    # Drop existing label columns before merging to avoid duplicates
                    label_cols_to_drop = [c for c in LABEL_COLS if c in df.columns]
                    if label_cols_to_drop:
                         df = df.drop(columns=label_cols_to_drop)
                         print(f"[LABEL] Dropped existing label columns before re-merge: {label_cols_to_drop}")

                    # Ensure store/menu columns are present and correctly formatted for merge
                    if 'store_menu' in df.columns and 'store' not in df.columns or 'menu' not in df.columns:
                         tmp = df['store_menu'].astype(str).str.split('_', n=1, expand=True)
                         df['store'] = tmp[0].astype(str).str.strip()
                         df['menu']  = tmp[1].fillna('').astype(str).str.strip()
                         print("[LABEL] Re-created 'store','menu' for label re-merge.")

                    df['store'] = df['store'].astype(str).str.strip()
                    df['menu']  = df['menu'].astype(str).str.strip()

                    merged = df.merge(
                         labels_df[['store','menu'] + LABEL_COLS],
                         on=['store','menu'],
                         how='left',
                         validate='m:1',
                         suffixes=('', '_dup')
                    )

                    # Handle unmatched labels (as in the original monkey-patch)
                    unmatched_mask = merged['menu_cluster'].isna() | merged['menu_cluster_label'].isna() | \
                                     merged['pattern_group'].isna() | merged['pattern_group_label'].isna()
                    n_unmatched = int(unmatched_mask.sum())
                    if n_unmatched > 0:
                         sample = merged.loc[unmatched_mask, ['store','menu']].drop_duplicates().head(10)
                         msg = f"[LABEL WARNING] Re-merge 라벨 미매칭 {n_unmatched}행. 예시:\n{sample}"
                         if STRICT_LABEL_CHECK:
                             raise ValueError(msg)
                         else:
                             print(msg)
                    else:
                         print("[LABEL] 라벨 재병합 완료 (모든 행 매칭).")

                    df = merged # Update df to the merged result
                    print("[LABEL] 라벨 재병합된 DataFrame 업데이트 완료.")

                    # Reapply column order logic
                    desired_order = [
                        'date_ordinal', 'date', 'store', 'menu', 'store_menu',
                        'menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label',
                        'sales'
                    ]
                    current_cols = df.columns.tolist()
                    ordered_cols = [col for col in desired_order if col in current_cols]
                    remaining_cols = [col for col in current_cols if col not in desired_order]
                    final_col_order = ordered_cols + remaining_cols
                    df = df[final_col_order]
                    print("[LABEL] 최종 컬럼 순서 재조정 완료 후 재저장.")


                # Save the final extended and featured DataFrame
                df.to_csv(output_file_path, index=False, encoding='utf-8')
                print(f"Saved final extended and featured {file_name}")

            else:
                print(f"No data to extend for {file_name}.")


        except Exception as e:
            print(f"Error extending test file {file_name}: {e}")


    processed_files_count += 1

print(f"\n모든 파일 처리 시도 완료. 총 {processed_files_count}개의 파일 처리가 완료되었습니다.")


Processing TEST_06.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2025-01-12,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_06.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_06.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_06.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending test file: TEST_06.csv
Loaded processed test file: TEST_06.csv
Original shape before extension: (5404, 26)
Extended TEST_06.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Applying add_domain_features to the extended DataFrame...
Applied add_domain_features. New shape: (6755, 26)
[LABEL] Re-merging labels after adding domain features...
[LABEL] Dropped existing label columns before re-merge: ['menu_cluster', 'menu_clu

,영업일자,영업장명_메뉴명,매출수량
0,2024-09-29,느티나무 셀프BBQ_1인 수저세트,5


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_03.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_03.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_03.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending test file: TEST_03.csv
Loaded processed test file: TEST_03.csv
Original shape before extension: (5404, 26)
Extended TEST_03.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Applying add_domain_features to the extended DataFrame...
Applied add_domain_features. New shape: (6755, 26)
[LABEL] Re-merging labels after adding domain features...
[LABEL] Dropped existing label columns before re-merge: ['menu_cluster', 'menu_clu

...

Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_02.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_02.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_02.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending test file: TEST_02.csv
Loaded processed test file: TEST_02.csv
Original shape before extension: (5404, 26)
Extended TEST_02.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Applying add_domain_features to the extended DataFrame...
Applied add_domain_features. New shape: (6755, 26)
[LABEL] Re-merging labels after adding domain features...
[LABEL] Dropped existing label columns before re-merge: ['menu_cluster', 'menu_clu

,영업일자,영업장명_메뉴명,매출수량
0,2025-04-27,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_09.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_09.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_09.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending test file: TEST_09.csv
Loaded processed test file: TEST_09.csv
Original shape before extension: (5404, 26)
Extended TEST_09.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Applying add_domain_features to the extended DataFrame...
Applied add_domain_features. New shape: (6755, 26)
[LABEL] Re-merging labels after adding domain features...
[LABEL] Dropped existing label columns before re-merge: ['menu_cluster', 'menu_clu

,영업일자,영업장명_메뉴명,매출수량
0,2023-01-01,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate train.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate train.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/train.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

모든 파일 처리 시도 완료. 총 11개의 파일 처리가 완료되었습니다.


## Handle 'sales' column

### Subtask:
For the extended dates in the test datasets, ensure the 'sales' column contains placeholder values (e.g., NaN or 0) as these are future predictions.


**Reasoning**:
Load the processed test DataFrame, identify the extended dates, and set the 'sales' column for these rows to NaN. Then save the updated DataFrame.



In [75]:
# File processing and saving loop
processed_files_count = 0
for file_name in csv_files:
    original_file_path = os.path.join(original_folder_path, file_name)
    output_file_path = os.path.join(output_folder_path, file_name) # Save with original filename

    # Check if the file is a test file
    is_test_file = file_name.startswith('TEST_')

    # Process the file (including the original steps + calendar features + label merge)
    # The monkey-patched process_csv_file already handles the initial processing and label merge
    process_csv_file(original_file_path, output_file_path)

    if is_test_file:
        print(f"\nExtending and finalizing test file: {file_name}")
        try:
            # Load the already processed DataFrame for the test file
            # Reloading to ensure the previous processing steps are included
            df = pd.read_csv(output_file_path, encoding='utf-8')
            print(f"Loaded processed test file: {file_name}")
            print(f"Original shape before extension: {df.shape}")

            # Ensure 'date' is datetime and 'date_ordinal' is correct
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df['date_ordinal'] = df['date'].map(lambda x: x.toordinal() if pd.notnull(x) else None)

            # Identify unique store/menu combinations
            unique_combinations = df[['store', 'menu']].drop_duplicates()

            extended_rows = []

            # For each unique combination, find the last date and generate next 7 days
            # and identify the original max date for later filtering
            original_max_dates = {}
            for index, row in unique_combinations.iterrows():
                store = row['store']
                menu = row['menu']

                # Filter for the current store/menu combination
                subset_df = df[(df['store'] == store) & (df['menu'] == menu)].copy()

                # Find the last date for this combination
                if not subset_df.empty:
                    last_date = subset_df['date'].max()
                    original_max_dates[(store, menu)] = last_date

                    # Generate the next 7 dates
                    next_dates = [last_date + timedelta(days=i) for i in range(1, 8)]

                    # Create new rows for these dates
                    for next_date in next_dates:
                        new_row = {
                            'date': next_date,
                            'date_ordinal': next_date.toordinal(),
                            'store': store,
                            'menu': menu,
                            'store_menu': f"{store}_{menu}",
                            'sales': np.nan # Sales is unknown for future dates
                        }
                        extended_rows.append(new_row)

            if extended_rows:
                # Create a DataFrame from the new rows
                extended_df = pd.DataFrame(extended_rows)

                # Concatenate the new rows to the original DataFrame
                df = pd.concat([df, extended_df], ignore_index=True)

                # Ensure date and date_ordinal are correct after concatenation
                df['date'] = pd.to_datetime(df['date'], errors='coerce')
                df['date_ordinal'] = df['date'].map(lambda x: x.toordinal() if pd.notnull(x) else None)

                print(f"Extended {file_name} by 7 days for each store/menu combination.")
                print(f"Shape after extension: {df.shape}")

                # Apply the add_domain_features function to the extended dataframe
                print("Applying add_domain_features to the extended DataFrame...")
                df = add_domain_features(df, train_peak_lut=None)
                print(f"Applied add_domain_features. New shape: {df.shape}")

                # Reapply labels to the combined data
                if MERGE_LABELS_ENABLED and labels_df is not None:
                    print("[LABEL] Re-merging labels after adding domain features...")
                    label_cols_to_drop = [c for c in LABEL_COLS if c in df.columns]
                    if label_cols_to_drop:
                         df = df.drop(columns=label_cols_to_drop)
                         print(f"[LABEL] Dropped existing label columns before re-merge: {label_cols_to_drop}")

                    if 'store_menu' in df.columns and ('store' not in df.columns or 'menu' not in df.columns):
                         tmp = df['store_menu'].astype(str).str.split('_', n=1, expand=True)
                         df['store'] = tmp[0].astype(str).str.strip()
                         df['menu']  = tmp[1].fillna('').astype(str).str.strip()
                         print("[LABEL] Re-created 'store','menu' for label re-merge.")

                    df['store'] = df['store'].astype(str).str.strip()
                    df['menu']  = df['menu'].astype(str).str.strip()

                    merged = df.merge(
                         labels_df[['store','menu'] + LABEL_COLS],
                         on=['store','menu'],
                         how='left',
                         validate='m:1',
                         suffixes=('', '_dup')
                    )

                    unmatched_mask = merged['menu_cluster'].isna() | merged['menu_cluster_label'].isna() | \
                                     merged['pattern_group'].isna() | merged['pattern_group_label'].isna()
                    n_unmatched = int(unmatched_mask.sum())
                    if n_unmatched > 0:
                         sample = merged.loc[unmatched_mask, ['store','menu']].drop_duplicates().head(10)
                         msg = f"[LABEL WARNING] Re-merge 라벨 미매칭 {n_unmatched}행. 예시:\n{sample}"
                         if STRICT_LABEL_CHECK:
                             raise ValueError(msg)
                         else:
                             print(msg)
                    else:
                         print("[LABEL] 라벨 재병합 완료 (모든 행 매칭).")

                    df = merged
                    print("[LABEL] 라벨 재병합된 DataFrame 업데이트 완료.")

                    # Reapply column order logic
                    desired_order = [
                        'date_ordinal', 'date', 'store', 'menu', 'store_menu',
                        'menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label',
                        'sales'
                    ]
                    current_cols = df.columns.tolist()
                    ordered_cols = [col for col in desired_order if col in current_cols]
                    remaining_cols = [col for col in current_cols if col not in desired_order]
                    final_col_order = ordered_cols + remaining_cols
                    df = df[final_col_order]
                    print("[LABEL] 최종 컬럼 순서 재조정 완료 후 재저장.")


                # --- Subtask 4: Set sales to NaN for extended dates ---
                print("Setting 'sales' to NaN for extended dates...")
                # Identify rows where the date is after the original max date for that store/menu combination
                def is_extended_date(row):
                    store = row['store']
                    menu = row['menu']
                    date = row['date']
                    original_max_date = original_max_dates.get((store, menu))
                    return original_max_date is not None and date > original_max_date

                extended_dates_mask = df.apply(is_extended_date, axis=1)
                df.loc[extended_dates_mask, 'sales'] = np.nan
                print(f"Set {extended_dates_mask.sum()} rows with extended dates to NaN in 'sales'.")
                # --- End Subtask 4 ---


                # Save the final extended and featured DataFrame
                df.to_csv(output_file_path, index=False, encoding='utf-8')
                print(f"Saved final extended and featured {file_name}")

            else:
                print(f"No data to extend for {file_name}.")


        except Exception as e:
            print(f"Error extending test file {file_name}: {e}")


    processed_files_count += 1

print(f"\n모든 파일 처리 시도 완료. 총 {processed_files_count}개의 파일 처리가 완료되었습니다.")


Processing TEST_06.csv...
Original shape: (5404, 3)
First row (for debugging):


,영업일자,영업장명_메뉴명,매출수량
0,2025-01-12,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_06.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_06.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_06.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending and finalizing test file: TEST_06.csv
Loaded processed test file: TEST_06.csv
Original shape before extension: (5404, 26)
Extended TEST_06.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Applying add_domain_features to the extended DataFrame...
Applied add_domain_features. New shape: (6755, 26)
[LABEL] Re-merging labels after adding domain features...
[LABEL] Dropped existing label columns before re-merge: ['menu_clus

,영업일자,영업장명_메뉴명,매출수량
0,2024-09-29,느티나무 셀프BBQ_1인 수저세트,5


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_03.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_03.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_03.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending and finalizing test file: TEST_03.csv
Loaded processed test file: TEST_03.csv
Original shape before extension: (5404, 26)
Extended TEST_03.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Applying add_domain_features to the extended DataFrame...
Applied add_domain_features. New shape: (6755, 26)
[LABEL] Re-merging labels after adding domain features...
[LABEL] Dropped existing label columns before re-merge: ['menu_clus

...

Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_02.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_02.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_02.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending and finalizing test file: TEST_02.csv
Loaded processed test file: TEST_02.csv
Original shape before extension: (5404, 26)
Extended TEST_02.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Applying add_domain_features to the extended DataFrame...
Applied add_domain_features. New shape: (6755, 26)
[LABEL] Re-merging labels after adding domain features...
[LABEL] Dropped existing label columns before re-merge: ['menu_clus

,영업일자,영업장명_메뉴명,매출수량
0,2025-04-27,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate TEST_09.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate TEST_09.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/TEST_09.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

Extending and finalizing test file: TEST_09.csv
Loaded processed test file: TEST_09.csv
Original shape before extension: (5404, 26)
Extended TEST_09.csv by 7 days for each store/menu combination.
Shape after extension: (6755, 26)
Applying add_domain_features to the extended DataFrame...
Applied add_domain_features. New shape: (6755, 26)
[LABEL] Re-merging labels after adding domain features...
[LABEL] Dropped existing label columns before re-merge: ['menu_clus

,영업일자,영업장명_메뉴명,매출수량
0,2023-01-01,느티나무 셀프BBQ_1인 수저세트,0


Columns renamed.
'store_menu' column split into 'store' and 'menu'.
'date' column converted to datetime and 'date_ordinal' column added.
Saving intermediate train.csv to /content/gdrive/My Drive/data_filtering/filtered...
Saved intermediate train.csv
[OK] v3 calendar features added. Columns: 22 → /content/gdrive/My Drive/data_filtering/filtered/train.csv
[LABEL] 라벨 병합 완료 (모든 행 매칭).
[LABEL] 추가된 라벨 컬럼: ['menu_cluster', 'menu_cluster_label', 'pattern_group', 'pattern_group_label']
[LABEL] 최종 컬럼 순서 재조정 완료.
[LABEL] 병합 결과 저장 완료.

모든 파일 처리 시도 완료. 총 11개의 파일 처리가 완료되었습니다.


## Save extended data

### Subtask:
Save the extended DataFrame with the added features to the output folder, ensuring the original filename is maintained.


## Summary:

### Data Analysis Key Findings

*   The test files, identified by the `TEST_` prefix, were successfully processed.
*   For each test file, the dataset was extended by 7 days for every unique store/menu combination.
*   The `date_ordinal` was correctly calculated for these new dates.
*   Calendar and domain-specific features were generated for the extended dates using the `add_domain_features` function.
*   The 'sales' column for the 7 extended days for each store/menu combination was explicitly set to `np.nan`, totaling 1351 rows per test file.
*   The extended and featured test DataFrames were saved to the output folder, retaining their original filenames.

### Insights or Next Steps

*   The extended test datasets are now ready for use in a forecasting model to predict future sales for the next 7 days.
*   Further analysis could involve inspecting the generated features for the extended dates to ensure their validity and relevance for forecasting.
